<a href="https://colab.research.google.com/github/rozaxa/Artificial-Intelligence-Workshop-II/blob/optimizer_comparison/optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [82]:
import ray
from ray import tune
from ray.air import session
from ray.tune.schedulers import ASHAScheduler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
import optuna
from ray import tune
from ray.tune.schedulers import ASHAScheduler

In [83]:
pip install optuna

In [84]:
pip install ray

In [85]:
!pip install ray[tune] tensorboardx


# Optuna




In [86]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)

data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

X = pd.DataFrame(data, columns=[
    "CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT"
])
y = pd.Series(target, name="MEDV")

In [87]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [88]:
baseline_model = RandomForestRegressor(random_state=42)
baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Baseline Regression RMSE:", baseline_rmse)

Baseline Regression RMSE: 2.8109631609391226


In [89]:
def objective_regression(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 3, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    reg_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    reg_model.fit(X_train, y_train)
    y_pred = reg_model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, y_pred))

In [90]:
study = optuna.create_study(direction="minimize")
study.optimize(objective_regression, n_trials=30)

print("Best parameters (Regression):", study.best_params)

[I 2024-12-05 16:48:28,841] A new study created in memory with name: no-name-9d5d102e-4b11-4a42-b774-8c441d6ec48b
[I 2024-12-05 16:48:29,972] Trial 0 finished with value: 3.153945587542294 and parameters: {'n_estimators': 294, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 0 with value: 3.153945587542294.
[I 2024-12-05 16:48:30,467] Trial 1 finished with value: 3.495625542421093 and parameters: {'n_estimators': 149, 'max_depth': 24, 'min_samples_split': 3, 'min_samples_leaf': 5}. Best is trial 0 with value: 3.153945587542294.
[I 2024-12-05 16:48:30,797] Trial 2 finished with value: 3.049483058222838 and parameters: {'n_estimators': 52, 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 2 with value: 3.049483058222838.
[I 2024-12-05 16:48:32,532] Trial 3 finished with value: 3.027163943965612 and parameters: {'n_estimators': 140, 'max_depth': 24, 'min_samples_split': 8, 'min_samples_leaf': 1}. Best is trial 3 with value: 3.02716

Best parameters (Regression): {'n_estimators': 50, 'max_depth': 28, 'min_samples_split': 3, 'min_samples_leaf': 1}


In [91]:
print("Best parameters (Regression):", study.best_params)

Best parameters (Regression): {'n_estimators': 50, 'max_depth': 28, 'min_samples_split': 3, 'min_samples_leaf': 1}


In [92]:
best_params = study.best_params
optimized_model = RandomForestRegressor(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    random_state=42
)
optimized_model.fit(X_train, y_train)
y_pred_optimized = optimized_model.predict(X_test)
optimized_rmse = np.sqrt(mean_squared_error(y_test, y_pred_optimized))

print("Optimized Regression RMSE:", optimized_rmse)

Optimized Regression RMSE: 2.681476689534341


In [93]:
print("Baseline Regression RMSE:", baseline_rmse)
print("Optimized Regression RMSE:", optimized_rmse)

Baseline Regression RMSE: 2.8109631609391226
Optimized Regression RMSE: 2.681476689534341


# Ray Tune

In [94]:
ray.init(ignore_reinit_error=True, num_cpus=2)

2024-12-05 16:48:58,136	INFO worker.py:1654 -- Calling ray.init() again after it has already been called.


Python version:,3.10.12
Ray version:,2.40.0


In [95]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]
data = pd.read_csv(url, header=None, names=columns, na_values="?")
data.dropna(inplace=True)



In [96]:
label_enc = LabelEncoder()
data["income"] = label_enc.fit_transform(data["income"])
X = pd.get_dummies(data.drop("income", axis=1), drop_first=True)
y = data["income"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [97]:
default_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
default_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [98]:
y_pred_default = default_model.predict(X_test)
default_accuracy = accuracy_score(y_test, y_pred_default)
print("Dokładność modelu przed optymalizacją:", default_accuracy)

Dokładność modelu przed optymalizacją: 0.8582834331337326


In [99]:
def train_classifier(config):
    clf = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=42
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    session.report({"mean_accuracy": acc})

config = {
    "n_estimators": tune.randint(50, 200),
    "max_depth": tune.randint(3, 20)
}

scheduler = ASHAScheduler(metric="mean_accuracy", mode="max")


In [100]:
analysis = tune.run(
    train_classifier,
    config=config,
    num_samples=10,
    scheduler=scheduler,
    resources_per_trial={"cpu": 1, "gpu": 0}
)

best_config = analysis.get_best_config(metric="mean_accuracy", mode="max")
print("Best parameters (Classification):", best_config)

+-------------------------------------------------------------------------+
| Configuration for experiment     train_classifier_2024-12-05_16-49-02   |
+-------------------------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator                  |
| Scheduler                        AsyncHyperBandScheduler                |
| Number of trials                 10                                     |
+-------------------------------------------------------------------------+

View detailed results here: /root/ray_results/train_classifier_2024-12-05_16-49-02
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2024-12-05_16-32-14_862485_584/artifacts/2024-12-05_16-49-02/train_classifier_2024-12-05_16-49-02/driver_artifacts`

Trial status: 10 PENDING
Current time: 2024-12-05 16:49:03. Total running time: 0s
Logical resource usage: 0/2 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:T4)
+----------------------

2024-12-05 16:49:45,298	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/train_classifier_2024-12-05_16-49-02' in 0.0194s.



Trial train_classifier_d5190_00009 finished iteration 1 at 2024-12-05 16:49:45. Total running time: 42s
+-------------------------------------------------------+
| Trial train_classifier_d5190_00009 result             |
+-------------------------------------------------------+
| checkpoint_dir_name                                   |
| time_this_iter_s                              2.88721 |
| time_total_s                                  2.88721 |
| training_iteration                                  1 |
| mean_accuracy                                 0.78858 |
+-------------------------------------------------------+

Trial train_classifier_d5190_00009 completed after 1 iterations at 2024-12-05 16:49:45. Total running time: 42s

Trial status: 10 TERMINATED
Current time: 2024-12-05 16:49:45. Total running time: 42s
Logical resource usage: 1.0/2 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:T4)
+------------------------------------------------------------------------------------------------

In [101]:
final_model = RandomForestClassifier(
    n_estimators=best_config["n_estimators"],
    max_depth=best_config["max_depth"],
    random_state=42
)
final_model.fit(X_train, y_train)

y_pred_final = final_model.predict(X_test)
final_accuracy = accuracy_score(y_test, y_pred_final)
print("Final accuracy of the model after optimization:", final_accuracy)

Final accuracy of the model after optimization: 0.8628896054045755


In [102]:
print("\nComparison of results:")
print(f"Accuracy before optimization: {default_accuracy}")
print(f"Accuracy after optimization: {final_accuracy}")


Comparison of results:
Accuracy before optimization: 0.8582834331337326
Accuracy after optimization: 0.8628896054045755
